# 05b — Patch No-Back

Replaces back-direction photos in `sampled_legs_directional.csv` with a better leg
from the same intersection.

**Why:** The back camera captures the car's trunk rather than the road ahead, making
photos unusable for intersection analysis. This happens when the survey vehicle drove
*away* from the intersection — the back camera happened to face the intersection, but
the resulting image shows the car boot.

**Strategy per back row (in order of preference):**
1. Find another leg of the same intersection in `leg_photo_selection_directional.csv`
   with `selected_direction != 'back'`. Prefer `front` > `left`/`right`; tiebreak on
   shortest `photo_dist_m`.
2. If *all* legs of that intersection are back: keep original, flag as `'unfixable'`.

**Input:**
- `data/processed/sampled_legs_directional.csv` — 368 rows, one per intersection (notebook 05)
- `data/processed/leg_photo_selection_directional.csv` — all legs for all intersections (notebook 04)

**Output:**
- `data/processed/sampled_legs_directional_noback.csv` — same 368 rows with back legs replaced;
  added column `direction_patched`: `False` / `True` / `'unfixable'`

**Depends on:** notebooks 04 and 05 must be run first.

In [1]:
import os
import pandas as pd

PROJECT_DIR   = r"C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections"
PROCESSED_DIR = os.path.join(PROJECT_DIR, "data", "processed")

# --- Image mode --- must match NB05
IMAGE_MODE    = "directional"  # change to "panorama" to process 360° panoramas

SAMPLED_CSV   = os.path.join(PROCESSED_DIR, f"sampled_legs_{IMAGE_MODE}.csv")
SELECTION_CSV = os.path.join(PROCESSED_DIR, f"leg_photo_selection_{IMAGE_MODE}.csv")
OUTPUT_CSV    = os.path.join(PROCESSED_DIR, f"sampled_legs_{IMAGE_MODE}_noback.csv")

# Columns that are intersection-level (not in the selection file).
# These are preserved from the original sampled row when a leg is swapped.
INTERSECTION_COLS = ["dim_type", "dim_risk", "dim_priority", "dim_speed", "is_centrum"]

# Direction preference when picking a replacement leg (lower = better)
DIRECTION_PREFERENCE = {"front": 0, "left": 1, "right": 2, "back": 3}

print(f"Sampled CSV  : {SAMPLED_CSV}")
print(f"Selection CSV: {SELECTION_CSV}")
print(f"Output CSV   : {OUTPUT_CSV}")

Sampled CSV  : C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections\data\processed\sampled_legs_directional.csv
Selection CSV: C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections\data\processed\leg_photo_selection_directional.csv
Output CSV   : C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections\data\processed\sampled_legs_directional_noback.csv


In [2]:
sampled   = pd.read_csv(SAMPLED_CSV)
selection = pd.read_csv(SELECTION_CSV)

print(f"Sampled legs:   {len(sampled)} rows")
print(f"Direction breakdown (before patch):")
print(sampled["selected_direction"].value_counts().to_string())
print()
print(f"Full selection: {len(selection)} rows across {selection['intersection_id'].nunique()} intersections")

Sampled legs:   368 rows
Direction breakdown (before patch):
selected_direction
front    261
back      95
left       7
right      5

Full selection: 12677 rows across 4615 intersections


In [3]:
def best_replacement(candidates: pd.DataFrame):
    """Return the best non-back row from a set of candidate legs, or None if all are back."""
    non_back = candidates[candidates["selected_direction"] != "back"].copy()
    if non_back.empty:
        return None
    # Sort by direction preference, then by proximity to intersection
    non_back["_dir_rank"] = non_back["selected_direction"].map(DIRECTION_PREFERENCE).fillna(99)
    non_back = non_back.sort_values(["_dir_rank", "photo_dist_m"])
    return non_back.iloc[0].drop(labels=["_dir_rank"])


# Index the full selection file by intersection_id for fast lookup
selection_by_intersection = {
    iid: grp for iid, grp in selection.groupby("intersection_id")
}

# object dtype so the column can hold False, True, and 'unfixable' together
sampled["direction_patched"] = pd.array([False] * len(sampled), dtype=object)

rows_fixed     = 0
rows_unfixable = 0

for idx, row in sampled.iterrows():
    if row["selected_direction"] != "back":
        continue

    all_legs    = selection_by_intersection.get(row["intersection_id"], pd.DataFrame())
    replacement = best_replacement(all_legs)

    if replacement is None:
        # All legs for this intersection are back — flag and leave unchanged
        sampled.at[idx, "direction_patched"] = "unfixable"
        rows_unfixable += 1
        print(f"  UNFIXABLE  intersection {row['intersection_id']}  (all legs are back)")
        continue

    # Overwrite leg-level columns with the replacement; keep intersection-level columns
    for col in replacement.index:
        if col not in INTERSECTION_COLS:
            sampled.at[idx, col] = replacement[col]

    sampled.at[idx, "direction_patched"] = True
    rows_fixed += 1

print(f"\nResults:")
print(f"  Fixed (new leg used):  {rows_fixed}")
print(f"  Unfixable (kept back): {rows_unfixable}")
print(f"  Already OK:            {len(sampled) - rows_fixed - rows_unfixable}")
print(f"\nDirection breakdown (after patch):")
print(sampled["selected_direction"].value_counts().to_string())

  UNFIXABLE  intersection 177267087  (all legs are back)
  UNFIXABLE  intersection 181273011  (all legs are back)
  UNFIXABLE  intersection 183276077  (all legs are back)
  UNFIXABLE  intersection 185275064  (all legs are back)
  UNFIXABLE  intersection 189276040  (all legs are back)
  UNFIXABLE  intersection 192267054  (all legs are back)

Results:
  Fixed (new leg used):  89
  Unfixable (kept back): 6
  Already OK:            273

Direction breakdown (after patch):
selected_direction
front    349
left       8
back       6
right      5


In [4]:
sampled.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(sampled)} rows to {OUTPUT_CSV}")

Saved 368 rows to C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections\data\processed\sampled_legs_directional_noback.csv
